In [1]:
# ---------- CELL 0: locate the project (same in every notebook) ----------
import os, sys
from pathlib import Path

def find_root(marker="PROJECT_ROOT.marker"):
    here = Path.cwd().resolve()                      # start where this notebook runs
    for folder in [here, *here.parents]:             # go UP one level at a time
        if (folder / marker).exists():
            return folder
    if "google.colab" in sys.modules:                # Colab: look inside Google Drive
        from google.colab import drive
        drive.mount("/content/drive")
        for pattern in (marker, f"*/{marker}", f"*/*/{marker}", f"*/*/*/{marker}"):
            hits = sorted(Path("/content/drive/MyDrive").glob(pattern))
            if hits:
                return hits[0].parent
    raise FileNotFoundError("PROJECT_ROOT.marker not found - finish Day 1 first.")

ROOT = find_root()
os.chdir(ROOT / "notebooks")
RAW, CLEAN = ROOT/"data"/"raw", ROOT/"data"/"clean"
FEATURES, MODEL_OUT = ROOT/"data"/"features", ROOT/"data"/"model_outputs"
SQL_EXPORTS, BI_READY = ROOT/"data"/"sql_exports", ROOT/"data"/"bi_ready"
MODELS, REPORTS, DOCS = ROOT/"models", ROOT/"reports", ROOT/"docs"
FIGURES = REPORTS/"figures"
sys.path.append(str(ROOT/"src"))
print("Project root:", ROOT)

Mounted at /content/drive
Project root: /content/drive/MyDrive/Internmo/netflix_churn_project


In [3]:
from pathlib import Path

helper = ROOT / "src" / "cleaning_helpers.py"

print("Project root:", ROOT)
print("Helper path:", helper)
print("Exists:", helper.exists())

Project root: /content/drive/MyDrive/Internmo/netflix_churn_project
Helper path: /content/drive/MyDrive/Internmo/netflix_churn_project/src/cleaning_helpers.py
Exists: True


In [5]:
# ---------- CELL 1: load tables and the log from Day 2 ----------
import pandas as pd
import numpy as np
from cleaning_helpers import standardize_country, standardize_device, save_clean

customers     = pd.read_csv(CLEAN / "customers.csv", parse_dates=["registration_date"])
subscriptions = pd.read_csv(CLEAN / "subscriptions.csv",
                            parse_dates=["subscription_start_date", "subscription_end_date", "cancellation_date"])
AS_OF = pd.Timestamp((CLEAN / "as_of_date.txt").read_text().strip())

viewing  = pd.read_csv(RAW / "viewing_activity.csv")
payments = pd.read_csv(RAW / "payments.csv")
tickets  = pd.read_csv(RAW / "support_tickets.csv")
feedback = pd.read_csv(RAW / "customer_feedback.csv")

# carry on the log started yesterday
cleaning_log = pd.read_csv(DOCS / "cleaning_log.csv").to_dict("records")
def log(table, issue, rows_affected, action):
    cleaning_log.append({"table": table, "issue": issue, "rows_affected": int(rows_affected), "action": action})
    print(f"[{table}] {issue}: {int(rows_affected)} rows -> {action}")

print("As-of date:", AS_OF.date())
print("rows -> viewing", len(viewing), "| payments", len(payments), "| tickets", len(tickets), "| feedback", len(feedback))

As-of date: 2026-06-30
rows -> viewing 66422 | payments 110735 | tickets 6120 | feedback 5046


In [6]:
# ---------- CELL 2: viewing_activity ----------
viewing["viewing_date"] = pd.to_datetime(viewing["viewing_date"])

# 2a) device names
before = viewing["device_type"].nunique()
viewing["device_type"] = standardize_device(viewing["device_type"])
n_missing = viewing["device_type"].isna().sum()
viewing["device_type"] = viewing["device_type"].fillna("Unknown")
log("viewing_activity", f"device spellings ({before} variants)", len(viewing), "standardised to 5 devices")
log("viewing_activity", "missing device_type", n_missing, "set to 'Unknown'")

# 2b) login country names (same rules as customers.country)
viewing["login_location"] = standardize_country(viewing["login_location"])

# 2c) extreme sessions: normal sessions are under about 140 minutes, a few are 500-900 (typing errors / binge bots)
WATCH_CAP, SESSION_CAP = 300, 360
n = (viewing["watch_duration_minutes"] > WATCH_CAP).sum()
viewing["watch_duration_minutes"] = viewing["watch_duration_minutes"].clip(upper=WATCH_CAP)
log("viewing_activity", f"watch_duration > {WATCH_CAP} minutes (outliers)", n, f"capped at {WATCH_CAP}")

n = (viewing["session_duration_minutes"] > SESSION_CAP).sum()
viewing["session_duration_minutes"] = viewing["session_duration_minutes"].clip(upper=SESSION_CAP)
log("viewing_activity", f"session_duration > {SESSION_CAP} minutes (outliers)", n, f"capped at {SESSION_CAP}")

# 2d) a session cannot be shorter than the time actually watched
n = (viewing["session_duration_minutes"] < viewing["watch_duration_minutes"]).sum()
viewing["session_duration_minutes"] = np.maximum(viewing["session_duration_minutes"], viewing["watch_duration_minutes"])
log("viewing_activity", "session shorter than watch time", n, "session raised to equal watch time")

# 2e) blank completion_percentage: we do NOT invent a number. NULL is the honest answer and AVG() ignores it.
log("viewing_activity", "missing completion_percentage", viewing["completion_percentage"].isna().sum(), "left blank on purpose")

print(viewing["device_type"].value_counts().to_dict())
print("countries in login_location:", viewing["login_location"].nunique())

[viewing_activity] device spellings (23 variants): 66422 rows -> standardised to 5 devices
[viewing_activity] missing device_type: 664 rows -> set to 'Unknown'
[viewing_activity] watch_duration > 300 minutes (outliers): 161 rows -> capped at 300
[viewing_activity] session_duration > 360 minutes (outliers): 161 rows -> capped at 360
[viewing_activity] session shorter than watch time: 6068 rows -> session raised to equal watch time
[viewing_activity] missing completion_percentage: 531 rows -> left blank on purpose
{'Smart TV': 25132, 'Mobile': 19822, 'Laptop': 9034, 'Tablet': 7811, 'Gaming Console': 3959, 'Unknown': 664}
countries in login_location: 20


In [7]:
# ---------- CELL 3: recover missing country from login location ----------
most_common_login = (viewing.groupby("customer_id")["login_location"]
                     .agg(lambda s: s.mode().iat[0]))          # the country they log in from most often

# TEST: for customers whose country IS known, how often does the login country agree?
known = customers[customers["country"] != "Unknown"].copy()
known["login_country"] = known["customer_id"].map(most_common_login)
testable = known.dropna(subset=["login_country"])
agreement = (testable["country"] == testable["login_country"]).mean()
print(f"Agreement between country and most common login country: {agreement:.1%}")

# USE: only if the test is convincing (we require at least 90%)
if agreement >= 0.90:
    missing = customers["country"] == "Unknown"
    recovered = customers.loc[missing, "customer_id"].map(most_common_login)
    customers.loc[missing, "country"] = recovered.fillna("Unknown")
    log("customers", "missing country recovered from login_location", recovered.notna().sum(),
        f"filled (agreement test = {agreement:.1%})")
print("customers still 'Unknown':", (customers["country"] == "Unknown").sum())

Agreement between country and most common login country: 99.8%
[customers] missing country recovered from login_location: 409 rows -> filled (agreement test = 99.8%)
customers still 'Unknown': 0


In [8]:
# ---------- CELL 4: payments - resolve subscription_id ----------
payments["payment_date"] = pd.to_datetime(payments["payment_date"])

subs = subscriptions[["subscription_id", "customer_id", "subscription_start_date",
                      "subscription_end_date", "monthly_price"]].rename(
    columns={"subscription_id": "matched_subscription_id", "monthly_price": "plan_price_then"})

# every payment is paired with every subscription row of the same customer (at most 2)
pairs = payments[["payment_id", "customer_id", "payment_date"]].merge(subs, on="customer_id")

# a blank end date means "still running" -> treat it as a far-away date
end_or_future = pairs["subscription_end_date"].fillna(pd.Timestamp("2100-01-01"))
inside_window = (pairs["payment_date"] >= pairs["subscription_start_date"]) & (pairs["payment_date"] <= end_or_future)
matched = pairs[inside_window]

# on the exact day a plan changes, two windows touch. Keep the NEWER subscription.
matched = (matched.sort_values(["payment_id", "subscription_start_date"])
                  .drop_duplicates("payment_id", keep="last"))

assert len(matched) == len(payments), "some payments found no subscription!"
payments = (payments.drop(columns="subscription_id")
                    .merge(matched[["payment_id", "matched_subscription_id", "plan_price_then"]], on="payment_id", how="left")
                    .rename(columns={"matched_subscription_id": "subscription_id"}))
log("payments", "subscription_id blank on every row", len(payments), "resolved by customer_id + payment_date window")
print("payments with a subscription_id now:", payments["subscription_id"].notna().sum(), "of", len(payments))

[payments] subscription_id blank on every row: 110735 rows -> resolved by customer_id + payment_date window
payments with a subscription_id now: 110735 of 110735


In [9]:
# ---------- CELL 5: payments - odd amounts ----------
payments["amount_original"] = payments["amount"]               # always keep the original value

ratio = (payments["amount"] / payments["plan_price_then"]).round(2)
differs = (payments["amount"] - payments["plan_price_then"]).abs() > 0.01

flag = pd.Series("ok", index=payments.index)
flag[differs] = "other_mismatch"
flag[differs & (payments["amount"] == 0)] = "zero_amount"
flag[differs & (ratio == 0.5)] = "half_proration"
flag[differs & (ratio == 2)] = "double_charge"
flag[differs & (ratio == 11)] = "x11_entry_error"
payments["amount_flag"] = flag

print(payments["amount_flag"].value_counts())

# Business rule (write it in your log!): a payment must equal the plan price on its subscription.
payments.loc[differs, "amount"] = payments.loc[differs, "plan_price_then"]
log("payments", "amount differs from plan price", differs.sum(), "amount set to the plan price; original kept in amount_original")

# failed_payment_reason must exist only for failed payments
bad = ((payments["payment_status"] == "Failed") != payments["failed_payment_reason"].notna()).sum()
print("rows where status and failure reason disagree:", bad)

amount_flag
ok                 109623
zero_amount           250
x11_entry_error       230
double_charge         226
half_proration        206
other_mismatch        200
Name: count, dtype: int64
[payments] amount differs from plan price: 1112 rows -> amount set to the plan price; original kept in amount_original
rows where status and failure reason disagree: 0


In [10]:
# ---------- CELL 6: support_tickets ----------
# invalid dates such as 2025-13-45 become NaT (blank). We cannot guess the real date, so we do not try.
parsed = pd.to_datetime(tickets["ticket_date"], errors="coerce")
n_bad = parsed.isna().sum()
tickets["ticket_date"] = parsed
log("support_tickets", "invalid ticket_date (e.g. 2025-13-45)", n_bad, "set to blank (ticket kept, date unknown)")

# resolution time and CSAT are blank only for tickets that are not finished yet. Prove it.
open_tickets = tickets["ticket_status"].isin(["Pending", "Escalated"])
print("blank CSAT on finished tickets:", tickets.loc[~open_tickets, "customer_satisfaction_score"].isna().sum(), "(should be 0)")
log("support_tickets", "blank resolution_time / CSAT", tickets["customer_satisfaction_score"].isna().sum(),
    "expected: ticket still Pending/Escalated. Left blank")

tickets["customer_satisfaction_score"] = tickets["customer_satisfaction_score"].astype("Int64")
tickets["ticket_description"] = tickets["ticket_description"].str.strip()

[support_tickets] invalid ticket_date (e.g. 2025-13-45): 61 rows -> set to blank (ticket kept, date unknown)
blank CSAT on finished tickets: 0 (should be 0)
[support_tickets] blank resolution_time / CSAT: 1528 rows -> expected: ticket still Pending/Escalated. Left blank


In [11]:
# ---------- CELL 7: customer_feedback ----------
feedback["feedback_date"] = pd.to_datetime(feedback["feedback_date"])

feedback["sentiment_from_rating"] = np.select(
    [feedback["rating"] >= 4, feedback["rating"] == 3], ["Positive", "Neutral"], default="Negative")
feedback["sentiment_mismatch"] = (feedback["sentiment_label"] != feedback["sentiment_from_rating"]).astype(int)

n = feedback["sentiment_mismatch"].sum()
log("customer_feedback", "sentiment_label disagrees with the star rating", n,
    "kept original label, added sentiment_from_rating and sentiment_mismatch")
print(f"mismatch rate: {n / len(feedback):.1%}")
pd.crosstab(feedback["sentiment_label"], feedback["sentiment_from_rating"])

[customer_feedback] sentiment_label disagrees with the star rating: 263 rows -> kept original label, added sentiment_from_rating and sentiment_mismatch
mismatch rate: 5.2%


sentiment_from_rating,Negative,Neutral,Positive
sentiment_label,,,
Negative,1371,39,52
Neutral,29,1515,62
Positive,36,45,1897


In [12]:
# ---------- CELL 8: save ----------
save_clean(customers, "customers", CLEAN)               # overwrites Day 2's version (now with recovered countries)

payment_columns = ["payment_id", "customer_id", "subscription_id", "payment_date", "amount",
                   "payment_method", "payment_status", "failed_payment_reason", "amount_original", "amount_flag"]
save_clean(viewing,  "viewing_activity",  CLEAN)
save_clean(payments[payment_columns], "payments", CLEAN)
save_clean(tickets,  "support_tickets",   CLEAN)
save_clean(feedback, "customer_feedback", CLEAN)

pd.DataFrame(cleaning_log).to_csv(DOCS / "cleaning_log.csv", index=False)
print("cleaning log rows:", len(cleaning_log))

saved customers.csv                rows=  8,000  columns=12
saved viewing_activity.csv         rows= 66,422  columns=9
saved payments.csv                 rows=110,735  columns=10
saved support_tickets.csv          rows=  6,120  columns=11
saved customer_feedback.csv        rows=  5,046  columns=8
cleaning log rows: 23


In [13]:
# ---------- CELL 9: validation across all 9 clean files ----------
tables = {p.stem: pd.read_csv(p) for p in sorted(CLEAN.glob("*.csv"))}
print({name: len(df) for name, df in tables.items()})
assert len(tables) == 9, "expected 9 clean CSV files"

valid_customers = set(tables["customers"]["customer_id"])
for name in ["subscriptions", "viewing_activity", "payments", "support_tickets", "customer_feedback", "churn_labels"]:
    orphans = (~tables[name]["customer_id"].isin(valid_customers)).sum()
    assert orphans == 0, f"{name} has customers that do not exist"
assert tables["viewing_activity"]["content_id"].isin(tables["content"]["content_id"]).all()
assert tables["payments"]["subscription_id"].isin(tables["subscriptions"]["subscription_id"]).all()
assert tables["viewing_activity"]["device_type"].nunique() == 6          # 5 devices + 'Unknown'
assert tables["viewing_activity"]["watch_duration_minutes"].max() <= 300
for name in ["customers", "viewing_activity", "payments", "support_tickets", "customer_feedback"]:
    assert tables[name].iloc[:, 0].is_unique, f"{name} has duplicate ids"
print("All Day 3 checks passed. 9 clean tables are ready for MySQL.")

{'churn_labels': 8000, 'content': 500, 'customer_feedback': 5046, 'customers': 8000, 'payments': 110735, 'subscription_plans': 3, 'subscriptions': 8875, 'support_tickets': 6120, 'viewing_activity': 66422}
All Day 3 checks passed. 9 clean tables are ready for MySQL.


In [14]:
# ---------- CELL 10: before vs after ----------
before = pd.read_csv(DOCS / "data_audit_tables.csv").set_index("table")
after = pd.DataFrame({name: {"rows": len(df), "missing_cells": int(df.isna().sum().sum())} for name, df in tables.items()}).T
scoreboard = before[["rows", "missing_cells"]].join(after, lsuffix="_raw", rsuffix="_clean")
scoreboard

,rows_raw,missing_cells_raw,rows_clean,missing_cells_clean
table,,,,
churn_labels,8000,12018,8000,12018
content,500,0,500,0
customer_feedback,5046,0,5046,0
customers,8048,9517,8000,8000
payments,110735,214264,110735,103529
subscription_plans,3,0,3,0
subscriptions,8875,21768,8875,21768
support_tickets,6120,3056,6120,3117
viewing_activity,66422,1195,66422,531


In [15]:
# ---------- CELL 11: project log ----------
entry = """## Day 3 - Cleaning part 2
- viewing_activity: 23 device spellings -> 5 (+Unknown). login_location standardised. Watch capped at 300, session at 360, session >= watch.
- customers.country: missing values recovered from the customer's most common login country (agreement test passed).
- payments: subscription_id resolved for all 110,735 rows (customer + date window; newer plan wins on change day).
- payments: rows where amount != plan price were corrected to the plan price. amount_original and amount_flag kept.
- support_tickets: invalid dates -> blank. Missing CSAT only on Pending/Escalated (expected).
- customer_feedback: added sentiment_from_rating and sentiment_mismatch (original label kept).
- 9 clean files in data/clean. Full list of actions: docs/cleaning_log.csv.
"""
with open(DOCS / "project_log.md", "a") as f:
    f.write(entry + "\n")
print("log updated")

log updated
